# geosite.dat для российских маршрутов

Собирает `geosite.dat` с двумя списками:

| Список | Содержимое | Маршрут |
|:---|:---|:---|
| `ru-blocked` | домены, заблокированные в России | через прокси |
| `ru-forbidden` | домены, которые сами закрывают доступ российским адресам | через прокси |

Маршрут у обеих веток один, но обрабатываются они по-разному, и в этом весь
смысл разделения.

`ru-blocked` собирается из реестра Роскомнадзора — миллион с лишним доменов,
куда по закону попадают решения не только о цензуре, но и о казино, фишинге,
наркотиках и пиратстве. Этот объём нужно сократить примерно в сотню раз, и
главный инструмент сокращения — фильтр популярности по CrUX.

`ru-forbidden` курируется вручную сообществом: несколько тысяч записей,
собранных людьми, которые проверяли каждую. Через фильтр популярности эта ветка
**не** проходит. Причина прямая: CrUX знает, какие сайты посещают из России, а
сайт, закрывший доступ российским адресам, из российской статистики по
определению выпадает. Пропусти его через тот же фильтр — и ручная работа
сообщества будет вычищена как «непопулярное».

## Бюджет

Расширение `NEPacketTunnelProvider` на iOS ограничено 50 МБ на процесс целиком,
включая сетевой стек и буферы пакетов. Практический потолок для списка —
**10 000 доменов на обе ветки вместе**. Проверка стоит в последнем разделе и при
превышении печатает, каким порогом популярности бюджет добирается.

## Допущение о переехавших

Заблокированный в России сайт либо перестаёт работать, либо переезжает за
рубеж. Отсюда две проверки: живость и страна хостинга. Домен, который жив и
физически размещён в России, удаляется — проксировать его незачем. Домен в зоне
`.ru`, уехавший на зарубежный хостинг, **остаётся**: зона сама по себе
основанием для удаления не является, иначе вырежет ровно тех, ради кого список
собирается.

## Порядок этапов

Этапы отсортированы по стоимости: сначала операции над множествами, затем один
запрос в BigQuery, и только после него DNS. К моменту DNS-стадий ветка
`ru-blocked` сокращена примерно в сотню раз, поэтому проверки, которые заняли бы
часы, занимают секунды.

1. Загрузка источников
2. Нормализация и дедупликация
3. Фильтр популярности по CrUX — только `ru-blocked`
4. Живость DNS — только `ru-blocked`
5. Страна хостинга — обе ветки
6. Слой фидов — обе ветки
7. Классификатор — обе ветки
8. Google Safe Browsing — обе ветки
9. Сборка `geosite.dat`

## Формат правил

Все домены пишутся как правило типа `Domain` (значение 2 в protobuf-схеме).
Оно матчит и сам домен, и все его поддомены: запись `example.com` покрывает
`api.example.com` и `a.b.example.com`. Перечислять поддомены отдельно не нужно,
такие записи удаляются на этапе дедупликации.

## Подключение в xray

```json
"rules": [
  { "type": "field", "domain": ["geosite:ru-blocked", "geosite:ru-forbidden"],
    "outboundTag": "proxy" },
  { "type": "field", "network": "tcp,udp", "outboundTag": "direct" }
]
```

In [3]:
!pip install -q requests dnspython tqdm scikit-learn scipy google-cloud-bigquery db-dtypes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 3.5 MB/s eta 0:00:00


In [6]:
import bisect
import concurrent.futures as cf
import ipaddress
import os
import random
import re
import time
from urllib.parse import urlsplit

import requests
import dns.resolver
import numpy as np
import requests
from scipy.stats import gaussian_kde
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, precision_recall_curve, roc_auc_score
from sklearn.model_selection import train_test_split
from tqdm import tqdm

In [10]:
# Цель сборки: столько доменов суммарно должно остаться в обеих ветках.
MAX_DOMAINS = 10_000

BLOCKED, FORBIDDEN = "ru-blocked", "ru-forbidden"
BRANCHES = [BLOCKED, FORBIDDEN]

# Удалять ли из ru-forbidden по решению слоёв 6–8, или только сообщать о
# найденном. Измерено сквозным прогоном: слой фидов снимает с этой ветки около
# 1.4% (порнография, торрент-трекеры, telegra.ph), классификатор — около 6%,
# и среди снятого им попадаются 4pda.to, tiktokv.com, ytimg.com, adobe.io.
# Ветка курируемая, ложное удаление здесь дороже лишней записи; при False
# те же слои отработают и напечатают находки, но список не тронут.
FORBIDDEN_DROP = False

## 1. Загрузка источников

Ветка `ru-blocked` берётся с antifilter.download. Это выгрузка самого реестра:
данные предоставляются Роскомнадзором по постановлению Правительства № 1101,
обновление проверяется раз в полчаса.

Прежний источник `zapret-info/z-i` не используется — его дамп заморожен на
`Updated: 2025-10-01`, репозиторий перестал наполняться. Список `domains_all.lst`
из Re-filter тоже не используется, но по другой причине: он производный от того
же antifilter.download, только прошедший авторскую чистку по нескольким сотням
ключевых слов (`casino`, `slot`, `1win`, `kinogo` и подобные). Мы берём исходное
надмножество и чистим сами — разделы 6 и 7 решают ту же задачу, но обновляются
вместе с фидами, а не вручную.

Ветка `ru-forbidden` собирается из подборок, которые ведут люди: список
сообщества antifilter, `community.lst` из Re-filter, `no-russia-hosts`,
геоблок-категории itdoginfo и Internet-Helper, наборы сервисов vernette.
Пересечения между ними велики и снимаются дедупликацией.

In [11]:
SOURCES = {
    BLOCKED: {
        "antifilter": "https://antifilter.download/list/domains.lst",
    },
    FORBIDDEN: {
        "antifilter_community": "https://community.antifilter.download/list/domains.lst",
        "re_filter_community": "https://raw.githubusercontent.com/1andrevich/Re-filter-lists/refs/heads/main/community.lst",
        "dartraiden": "https://raw.githubusercontent.com/dartraiden/no-russia-hosts/refs/heads/master/hosts.txt",
        "itdoginfo": "https://raw.githubusercontent.com/itdoginfo/allow-domains/refs/heads/main/Categories/geoblock.lst",
        "internet_helper": "https://raw.githubusercontent.com/Internet-Helper/Unblock-for-Russia/refs/heads/main/geoblock.lst",
    },
}

VERNETTE_API_URL = "https://api.github.com/repos/vernette/rulesets/contents/raw"
VERNETTE_RAW_BASE = "https://raw.githubusercontent.com/vernette/rulesets/master/raw/"

HEADERS = {"User-Agent": "geosite-ru-builder"}
GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN")
if GITHUB_TOKEN:
    HEADERS["Authorization"] = f"Bearer {GITHUB_TOKEN}"

In [ ]:
def fetch_plain_list(url, timeout=300):
    """Читает список вида «один домен на строку»; строки с # и ! пропускаются."""
    r = requests.get(url, headers=HEADERS, timeout=timeout)
    r.raise_for_status()
    return [line.strip().lower() for line in r.text.splitlines()
            if line.strip() and not line.startswith(("#", "!"))]


HOSTS_LINE = re.compile(r"^0\.0\.0\.0\s+([a-z0-9.\-]+)$")


def fetch_hosts_file(url, timeout=300):
    """Читает hosts-формат: «0.0.0.0 domain». Нужен только для фидов раздела 6."""
    r = requests.get(url, headers=HEADERS, timeout=timeout)
    r.raise_for_status()
    out = []
    for line in r.text.splitlines():
        m = HOSTS_LINE.match(line.strip().lower())
        if m and m.group(1) != "0.0.0.0":
            out.append(m.group(1))
    return out

In [ ]:
def fetch_vernette_rulesets(timeout=120):
    """Все .txt из vernette/rulesets/raw.

    Список файлов запрашивается через API, а не хардкодится: набор сервисов
    в репозитории меняется. Берутся и сводный файл, и отдельные сервисные —
    сводный не является надмножеством, часть сервисов из него исключена
    автором намеренно. Файл rkn.txt пропускается: это копия реестра, то есть
    материал другой ветки.
    """
    resp = requests.get(VERNETTE_API_URL, headers=HEADERS, timeout=timeout)
    resp.raise_for_status()
    entries = resp.json()
    if isinstance(entries, dict):
        raise RuntimeError(entries.get("message", "неожиданный ответ API"))

    domains = set()
    for name in (e["name"] for e in entries
                 if e["name"].endswith(".txt") and e["name"] != "rkn.txt"):
        try:
            domains.update(fetch_plain_list(VERNETTE_RAW_BASE + name))
        except Exception as exc:
            print(f"  пропущен {name}: {exc}")
        time.sleep(0.2)
    return sorted(domains)

In [ ]:
raw = {name: [] for name in BRANCHES}

for _branch, _sources in SOURCES.items():
    for _name, _url in _sources.items():
        try:
            _got = fetch_plain_list(_url)
            raw[_branch].extend(_got)
            print(f"{_branch:13s} {_name:22s} {len(_got):>9,}")
        except Exception as exc:
            print(f"{_branch:13s} {_name:22s} недоступен — {exc}")

try:
    _got = fetch_vernette_rulesets()
    raw[FORBIDDEN].extend(_got)
    print(f"{FORBIDDEN:13s} {'vernette':22s} {len(_got):>9,}")
except Exception as exc:
    print(f"{FORBIDDEN:13s} {'vernette':22s} недоступен — {exc}")

print()
for _name in BRANCHES:
    print(f"{_name}: сырых записей {len(raw[_name]):,}")

ru-blocked    antifilter             1,588,175
ru-forbidden  antifilter_community         486
ru-forbidden  re_filter_community          699
ru-forbidden  dartraiden                   729
ru-forbidden  itdoginfo                    466
ru-forbidden  internet_helper              962
ru-forbidden  vernette                     798

ru-blocked: сырых записей 1,588,175
ru-forbidden: сырых записей 4,140


## 2. Нормализация и дедупликация

Источники записывают домены по-разному: с префиксом `www.`, с обёрткой
`*.example.com`, изредка с полным URL или портом. Нормализация приводит записи к
одному виду и отбрасывает то, что доменом не является.

Дедупликация идёт двумя проходами. Первый снимает точные повторы. Второй убирает
записи, уже покрытые правилом родительского домена: при наличии `example.com`
записи `sub.example.com` и `a.b.example.com` избыточны, так как правило типа
`Domain` матчит поддомены любой глубины. Построчное сравнение такие пары не
находит — строки не совпадают. На выгрузке реестра второй проход снимает
заметную долю: там записи вида `*.domain.tld` соседствуют с собственными
поддоменами.

Ветки обрабатываются раздельно и остаются раздельными до самой сборки.

Здесь же определяется `registrable()` — домен вместе с одной меткой сверх
публичного суффикса (`sub.example.co.uk` → `example.co.uk`). Он нужен и фильтру
по CrUX, и защите от ошибок классификатора, поэтому определяется один раз.

In [19]:
PSL_URL = "https://raw.githubusercontent.com/publicsuffix/list/master/public_suffix_list.dat"

_psl_text = requests.get(PSL_URL, headers=HEADERS, timeout=120).text

PSL_RULES, PSL_WILDCARDS, PSL_EXCEPTIONS = set(), set(), set()
for _line in _psl_text.splitlines():
    _rule = _line.strip()
    if not _rule or _rule.startswith("//"):
        continue
    if _rule.startswith("!"):
        PSL_EXCEPTIONS.add(_rule[1:])
    elif _rule.startswith("*."):
        PSL_WILDCARDS.add(_rule[2:])
    else:
        PSL_RULES.add(_rule)

print(f"правил в PSL: обычных {len(PSL_RULES):,}, "
      f"с шаблоном {len(PSL_WILDCARDS):,}, исключений {len(PSL_EXCEPTIONS):,}")


def public_suffix(domain):
    """Наиболее длинный публичный суффикс домена.

    Шаблонные правила PSL (`*.ck`) и исключения (`!www.ck`) учитываются:
    без них `example.ck` был бы разобран как регистрируемый домен, хотя
    регистрация в этой зоне идёт на уровень ниже.
    """
    labels = domain.split(".")
    for i in range(len(labels)):
        candidate = ".".join(labels[i:])
        if candidate in PSL_EXCEPTIONS:
            return ".".join(labels[i + 1:])
        if candidate in PSL_RULES:
            return candidate
        parent = ".".join(labels[i + 1:])
        if parent and parent in PSL_WILDCARDS:
            return candidate
    return labels[-1]


def registrable(domain):
    """Домен вместе с одной меткой сверх публичного суффикса."""
    suffix = public_suffix(domain)
    if domain == suffix:
        return domain
    head = domain[: -len(suffix) - 1]
    return head.rsplit(".", 1)[-1] + "." + suffix

правил в PSL: обычных 9,957, с шаблоном 283, исключений 8


In [ ]:
# Записи, которые доменами не являются или бесполезны в правиле маршрутизации.
JUNK_PATTERNS = [
    r"\.in-addr\.arpa$", r"\.ip6\.arpa$", r"\.(ip4|ip6)\.static\.",
    r"\.(rev|ptr)\.",
    r"\.(amazonaws|googleusercontent|cloudapp|compute)\.com$",
    r"^[a-f0-9]{16,}\.",
]
JUNK_RE = re.compile("|".join(JUNK_PATTERNS))

# Метка: буквы, цифры, дефис и подчёркивание (последнее встречается в
# служебных именах). Зона верхнего уровня — либо буквенная, либо punycode
# вида xn--p1ai, поэтому цифры и дефисы в ней допускаются, но начинается
# она с буквы. Общая длина имени ограничена 253 символами по RFC 1035.
DOMAIN_RE = re.compile(
    r"^(?=.{4,253}$)"
    r"(?:[a-z0-9_](?:[a-z0-9_\-]{0,61}[a-z0-9_])?\.)+"
    r"[a-z][a-z0-9\-]{1,62}$"
)


def normalize(entry):
    """Приводит запись источника к голому доменному имени.

    Возвращает None, если запись доменом не является. Снимаются: схема,
    путь, порт, обёртка `*.`, префикс `www.`, завершающая точка. Имена с
    не-ASCII символами переводятся в punycode: geosite.dat хранит домены в
    ASCII, а клиент сравнивает строки байт в байт.
    """
    entry = entry.strip().lower().strip(".")
    if not entry:
        return None
    if "//" in entry:
        entry = urlsplit(entry if "://" in entry else "http://" + entry).netloc
    entry = entry.split("/")[0].split("?")[0]
    if "@" in entry:
        entry = entry.rsplit("@", 1)[-1]
    entry = entry.split(":")[0].strip(".")
    if entry.startswith("*."):
        entry = entry[2:]
    if entry.startswith("www."):
        entry = entry[4:]
    if not entry or "." not in entry:
        return None
    if not entry.isascii():
        try:
            entry = entry.encode("idna").decode("ascii")
        except Exception:
            return None
    try:
        ipaddress.ip_address(entry)
        return None
    except ValueError:
        pass
    if not DOMAIN_RE.match(entry) or JUNK_RE.search(entry):
        return None
    return entry

In [ ]:
def dedup_exact(domains):
    seen, out = set(), []
    for domain in domains:
        if domain not in seen:
            seen.add(domain)
            out.append(domain)
    return out


def dedup_by_coverage(domains):
    """Удаляет домены, покрытые правилом уже принятого родителя.

    Сортировка по числу меток гарантирует, что родитель рассматривается
    раньше потомка, поэтому достаточно одного прохода.
    """
    kept, kept_set = [], set()
    for domain in sorted(set(domains), key=lambda d: (d.count("."), d)):
        labels = domain.split(".")
        if any(".".join(labels[i:]) in kept_set for i in range(1, len(labels))):
            continue
        kept.append(domain)
        kept_set.add(domain)
    return kept

In [ ]:
branch = {}

for _name in BRANCHES:
    _normalized = [d for d in (normalize(x) for x in raw[_name]) if d]
    _exact = dedup_exact(_normalized)
    branch[_name] = dedup_by_coverage(_exact)
    print(f"{_name}:")
    print(f"  сырых записей     {len(raw[_name]):>9,}")
    print(f"  не домены         {len(raw[_name]) - len(_normalized):>9,}")
    print(f"  точных повторов   {len(_normalized) - len(_exact):>9,}")
    print(f"  покрыто родителем {len(_exact) - len(branch[_name]):>9,}")
    print(f"  осталось          {len(branch[_name]):>9,}")

# Ветка ru-blocked до фильтра популярности понадобится в разделе 7 для
# калибровки порога классификатора: там нужна широкая популяция, а не остаток.
blocked_before_crux = list(branch[BLOCKED])

ru-blocked:
  сырых записей     1,588,175
  не домены               567
  точных повторов      12,620
  покрыто родителем   160,649
  осталось          1,414,339
ru-forbidden:
  сырых записей         4,140
  не домены                 3
  точных повторов       2,218
  покрыто родителем       274
  осталось              1,645


## 3. Фильтр популярности по CrUX — только `ru-blocked`

Самый отсекающий этап: в выгрузке реестра держатся сотни тысяч доменов, до
которых никто не доходит. CrUX (Chrome User Experience Report) публикует в
BigQuery срез по странам — какие origins реально посещались из России.

Что нужно знать о поле `experimental.popularity.rank`. Это не порядковый номер,
а корзина по порядку величины с полушагами: 1000, 5000, 10 000, 50 000,
100 000, 500 000, 1 000 000 и далее. Условие `rank <= 10000` означает «входит в
десять тысяч самых посещаемых», а не «занимает десятитысячное место»; внутри
корзины порядок не определён. Поэтому `CRUX_RANK_MAX` — ручка грубой настройки,
и осмысленные её значения перечислены в `CRUX_BUCKETS`.

Смещение, о котором стоит помнить. CrUX собирает данные с согласившихся
пользователей Chrome, страна определяется по адресу пользователя.
Заблокированный сайт, до которого доходят только через VPN, попадает в срез той
страны, где стоит выходной узел, — то есть частично выпадает из RU-среза. Фильтр
смещён в сторону «частично доступного»: полностью отрезанные ресурсы с малой
аудиторией он теряет. Это осознанный размен при бюджете в 10 000 записей, и
именно поэтому ветка `ru-forbidden` через него не идёт.

Датасет публикуется во второй вторник после отчётного месяца, поэтому последний
доступный месяц ищется запросом, а не вычисляется из календаря: в первую неделю
месяца арифметическая догадка промахивается. Пробный запрос читает единственный
столбец и стоит копейки.

In [ ]:
try:
    from google.colab import auth
    auth.authenticate_user()
except ImportError:
    pass  # вне Colab — Application Default Credentials от шага авторизации

from google.cloud import bigquery

# В Actions переменную выставляет google-github-actions/auth; значение справа
# остаётся для запуска в Colab.
PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "rkn-block-filtering")
bq_client = bigquery.Client(project=PROJECT_ID)
print(f"проект: {PROJECT_ID}")

In [ ]:
CRUX_TABLE = "chrome-ux-report.experimental.country"

# Осмысленные пороги — границы корзин CrUX. Промежуточные значения
# эквивалентны ближайшей нижней границе.
CRUX_BUCKETS = [1_000, 5_000, 10_000, 50_000, 100_000, 500_000, 1_000_000]
CRUX_RANK_MAX = 1_000_000

_probe = f"SELECT MAX(yyyymm) AS m FROM `{CRUX_TABLE}` WHERE yyyymm >= 202401"
CRUX_YYYYMM = int(bq_client.query(_probe).to_dataframe()["m"][0])
print(f"последний доступный срез CrUX: {CRUX_YYYYMM}")

последний доступный срез CrUX: 202607


In [ ]:
CRUX_QUERY = f"""
SELECT DISTINCT origin, experimental.popularity.rank AS rank
FROM `{CRUX_TABLE}`
WHERE yyyymm = {CRUX_YYYYMM}
  AND country_code = "ru"
  AND experimental.popularity.rank <= {CRUX_RANK_MAX}
"""

crux_ru = bq_client.query(CRUX_QUERY).to_dataframe()
print(f"origins в RU-срезе: {len(crux_ru):,}")

# origin имеет вид https://www.example.com — нужен только хост, и дальше
# сведённый к регистрируемому домену: правила всё равно работают на уровне
# домена вместе с поддоменами.
CRUX_RANK = {}
for _origin, _rank in zip(crux_ru["origin"], crux_ru["rank"]):
    _host = (urlsplit(_origin).hostname or "").lower().strip(".")
    if _host.startswith("www."):
        _host = _host[4:]
    if not _host:
        continue
    _key = registrable(_host)
    CRUX_RANK[_key] = min(_rank, CRUX_RANK.get(_key, _rank))

print(f"уникальных регистрируемых доменов: {len(CRUX_RANK):,}")

origins в RU-срезе: 639,155
уникальных регистрируемых доменов: 469,311


In [ ]:
_before = len(branch[BLOCKED])
branch[BLOCKED] = [d for d in branch[BLOCKED] if registrable(d) in CRUX_RANK]

print(f"{BLOCKED}: {_before:,} → {len(branch[BLOCKED]):,} "
      f"({len(branch[BLOCKED]) / _before:.1%})")
print(f"{FORBIDDEN}: {len(branch[FORBIDDEN]):,} (фильтр не применяется)")

# Распределение по корзинам показывает, каким порогом добирается бюджет.
print("\nнакопительно по корзинам ранга:")
for _bucket in CRUX_BUCKETS:
    _count = sum(1 for d in branch[BLOCKED] if CRUX_RANK[registrable(d)] <= _bucket)
    print(f"  rank <= {_bucket:>9,}: {_count:>7,}")

ru-blocked: 1,414,339 → 25,641 (1.8%)
ru-forbidden: 1,645 (фильтр не применяется)

накопительно по корзинам ранга:
  rank <=     1,000:     552
  rank <=     5,000:   1,366
  rank <=    10,000:   2,085
  rank <=    50,000:   6,343
  rank <=   100,000:  10,533
  rank <=   500,000:  19,964
  rank <= 1,000,000:  25,641


## 4. Живость домена — только `ru-blocked`

Проверка стоит здесь, а не в начале, потому что она сетевая и её стоимость
пропорциональна длине списка. На сотнях тысяч доменов проход занял бы часы;
после раздела 3 в ветке тысячи, и проход занимает секунды.

Ветка `ru-forbidden` проверку пропускает намеренно: сайт, закрывший доступ
российским адресам, по определению жив — он именно работает, просто не для нас.
Отдельного смысла подтверждать это запросом нет.

Отказы разделяются по типу. `NXDOMAIN` — имя не делегировано, домен мёртв,
перепроверять нечего. Таймаут или отказ всех серверов — состояние неизвестное:
это может быть перегруз резолвера или медленная зона, поэтому такие имена идут
во второй проход с увеличенным временем ожидания. Отсутствие A-записи
проверяется дополнительно по AAAA: домен может быть доступен только по IPv6.

Таймаут задаётся аргументом при каждом вызове, а не записывается в объект
резолвера: объект общий для всех потоков, и правка его полей во время работы
пула даёт разное поведение у разных потоков.

In [ ]:
RESOLVER = dns.resolver.Resolver()
RESOLVER.nameservers = ["1.1.1.1", "8.8.8.8", "9.9.9.9"]

FAST_TIMEOUT = 2.0
SLOW_TIMEOUT = 15.0


def resolve(name, rdtype, timeout=FAST_TIMEOUT):
    """Записи запрошенного типа; пустой список при любой ошибке."""
    try:
        answers = RESOLVER.resolve(name, rdtype, lifetime=timeout)
    except Exception:
        return []
    return [str(r) for r in answers]


def lookup_status(domain, timeout=FAST_TIMEOUT):
    """alive — отвечает; dead — не делегирован; unknown — не удалось выяснить."""
    try:
        RESOLVER.resolve(domain, "A", lifetime=timeout)
        return "alive"
    except dns.resolver.NXDOMAIN:
        return "dead"
    except dns.resolver.NoAnswer:
        return "alive" if resolve(domain, "AAAA", timeout) else "dead"
    except Exception:
        return "unknown"


def check_liveness(domains, timeout=FAST_TIMEOUT, workers=100, desc="DNS"):
    """Раскладывает домены по трём корзинам состояния."""
    result = {"alive": [], "dead": [], "unknown": []}
    with cf.ThreadPoolExecutor(max_workers=workers) as pool:
        statuses = pool.map(lambda d: lookup_status(d, timeout), domains)
        for domain, status in tqdm(zip(domains, statuses), total=len(domains), desc=desc):
            result[status].append(domain)
    return result

In [ ]:
_t0 = time.time()
_first = check_liveness(branch[BLOCKED], FAST_TIMEOUT, desc="DNS, первый проход")
print(f"живых {len(_first['alive']):,}, "
      f"не делегировано {len(_first['dead']):,}, "
      f"неясно {len(_first['unknown']):,}")

_second = check_liveness(_first["unknown"], SLOW_TIMEOUT,
                         desc="DNS, повтор с большим ожиданием")
print(f"из неясных ожили: {len(_second['alive']):,}")

_before = len(branch[BLOCKED])
branch[BLOCKED] = _first["alive"] + _second["alive"]
print(f"\n{BLOCKED}: {_before:,} → {len(branch[BLOCKED]):,}, {time.time() - _t0:.0f}с")

DNS, первый проход: 100%|██████████| 23692/23692 [00:23<00:00, 1013.90it/s]


живых 23,627, не делегировано 0, неясно 65


DNS, повтор с большим ожиданием: 100%|██████████| 65/65 [00:08<00:00,  7.66it/s]

из неясных ожили: 65

ru-blocked: 23,692 → 23,692, 38с


## 5. Страна хостинга — обе ветки

Домен, физически размещённый в России, удаляется. Для `ru-blocked` это значит,
что сайт странен из-за предположения, что заблокированный в России сайт не может продолжать хоститься в России, и запись в списке только расходует бюджет. Для
`ru-forbidden` — что запись залётная: российский сайт не может закрывать доступ
российским адресам, а попав в этот список, был бы отправлен через зарубежный
прокси.

Решение принимается **только по адресу сайта**, не по зоне. Зона `.ru` сама по
себе ничего не значит: заблокированный ресурс, переехавший на зарубежный
хостинг, обычно сохраняет прежнее имя, и удаление по зоне вырезало бы ровно тех,
ради кого список собирается.

По той же причине не используются адреса NS-серверов. Сайт, уехавший за рубеж,
часто остаётся на DNS-серверах российского регистратора, и признак «все NS в
России» дал бы те же ложные удаления. Сигнал по NS сильнее ловит российский
хостинг, но здесь важнее не потерять переехавших.

Удаление происходит, когда **все** полученные A-записи попадают в российские
диапазоны. Смешанный результат трактуется в пользу сохранения: так ведут себя
сети доставки контента с точкой присутствия в Москве, и сам факт такой точки не
означает доступности ресурса.

База диапазонов: `sapics/ip-location-db`, файл `server-country-ipv4.csv`,
лицензия PDDL, без ключей и лимитов на запросы.

In [ ]:
GEOIP_URL = "https://github.com/sapics/ip-location-db/releases/download/latest/server-country-ipv4.csv"


class CountryDB:
    """Диапазоны IPv4 → код страны, поиск двоичным делением."""

    def __init__(self, csv_text):
        rows = []
        for line in csv_text.splitlines():
            parts = line.split(",")
            if len(parts) < 3:
                continue
            try:
                start = int(ipaddress.IPv4Address(parts[0].strip()))
                end = int(ipaddress.IPv4Address(parts[1].strip()))
            except ipaddress.AddressValueError:
                continue  # заголовок или строка с IPv6
            rows.append((start, end, parts[2].strip().upper()))

        # Двоичный поиск требует упорядоченности; файл отсортирован, но
        # полагаться на это без проверки нельзя — при нарушении порядка
        # lookup молча вернёт неверную страну.
        rows.sort()
        self.starts = [r[0] for r in rows]
        self.ends = [r[1] for r in rows]
        self.codes = [r[2] for r in rows]

    def lookup(self, ip):
        try:
            value = int(ipaddress.IPv4Address(ip))
        except ipaddress.AddressValueError:
            return None
        idx = bisect.bisect_right(self.starts, value) - 1
        if idx >= 0 and value <= self.ends[idx]:
            return self.codes[idx]
        return None


geoip = CountryDB(requests.get(GEOIP_URL, headers=HEADERS, timeout=300).text)
print(f"диапазонов в базе: {len(geoip.starts):,}")

диапазонов в базе: 273,550


In [ ]:
def hosting_countries(domain, limit=3):
    """Коды стран A-записей домена."""
    return [geoip.lookup(ip) for ip in resolve(domain, "A")[:limit]]


def split_by_hosting(domains, workers=50, desc="Страна хостинга"):
    """Отделяет размещённые в России от остальных."""
    outside, inside = [], []
    with cf.ThreadPoolExecutor(max_workers=workers) as pool:
        countries = pool.map(hosting_countries, domains)
        for domain, codes in tqdm(zip(domains, countries), total=len(domains), desc=desc):
            (inside if codes and all(cc == "RU" for cc in codes) else outside).append(domain)
    return outside, inside

In [ ]:
ru_hosted = {}

for _name in BRANCHES:
    branch[_name], ru_hosted[_name] = split_by_hosting(
        branch[_name], desc=f"Страна хостинга, {_name}")
    print(f"{_name}: оставлено {len(branch[_name]):,}, "
          f"удалено как RU-хостинг {len(ru_hosted[_name]):,}")

# Контроль: в удалённых из ru-forbidden не должно быть зарубежных сервисов.
print(f"\nпример удалённых из {FORBIDDEN}:")
for _domain in ru_hosted[FORBIDDEN][:30]:
    print(f"  {_domain}")

Страна хостинга, ru-blocked: 100%|██████████| 23692/23692 [00:39<00:00, 595.28it/s]


ru-blocked: оставлено 22,431, удалено как RU-хостинг 1,261


Страна хостинга, ru-forbidden: 100%|██████████| 1645/1645 [00:03<00:00, 413.94it/s]

ru-forbidden: оставлено 1,639, удалено как RU-хостинг 6

пример удалённых из ru-forbidden:
  epg.one
  gpt3-openai.com
  habr.com
  iichan.hk
  php.su
  pushbr.com


## 6. Слой фидов — обе ветки

Реестр не содержит поля с причиной блокировки, поэтому категория определяется по
самому домену. Первый слой — сверка с готовыми категоризованными списками.

UT1 (Université Toulouse Capitole, зеркало обновляется раз в сутки, CC BY-SA)
раскладывает домены по 67 категориям; StevenBlack даёт отдельный список
порнографии; AdGuard ведёт свежие фишинговые поддомены на конструкторах сайтов.
Слой точный, но запаздывающий: UT1 пополняется вручную на 50–300 записей в день,
а мусорные домены реестра размножаются быстрее. Как единственный механизм он
непригоден, отсюда классификатор в разделе 7.

Категории `phishing` и `malware` в UT1 совпадают почти полностью и считаются
одним источником, берётся `malware`. Категории `vpn`, `redirector`, `doh`,
`shortener` в мусор **не** включаются: для школьного прокси, под который UT1
создавался, это нежелательный трафик, для этого списка — целевой.

Категория `adult` недоступна через raw.githubusercontent — файл превышает лимит
размера GitHub, поэтому порнография берётся из расширений StevenBlack.

Ветка `ru-forbidden` проходит слой наравне с `ru-blocked`. Замер сквозного
прогона: снимается около 1.4% ветки, и это не только мусор — вместе с
порнографией и торрент-трекерами под правило попадают `telegra.ph`,
`joyreactor.cc`, `bitcoin.org`. UT1 относит их к своим категориям по собственным
основаниям, для маршрутизации через прокси они целевые. Поэтому находки по этой
ветке печатаются целиком, а удаление управляется флагом `FORBIDDEN_DROP` из
второй ячейки.

In [ ]:
UT1_BASE = "https://raw.githubusercontent.com/olbat/ut1-blacklists/master/blacklists/{}/domains"

UT1_JUNK = ["gambling", "drogue", "cryptojacking", "ddos", "stalkerware", "malware"]
UT1_LEGIT = ["shopping", "bank", "press", "jobsearch", "games"]

PORN_URLS = [
    "https://raw.githubusercontent.com/StevenBlack/hosts/master/extensions/porn/sinfonietta/hosts",
    "https://raw.githubusercontent.com/StevenBlack/hosts/master/extensions/porn/clefspeare13/hosts",
]

ADGUARD_PHISHING_URL = "https://raw.githubusercontent.com/AdguardTeam/HostlistsRegistry/main/filters/security/filter_30_PhishingURLBlocklist/filter.txt"
ADBLOCK_RULE = re.compile(r"^\|\|([a-z0-9.\-]+)\^")


def fetch_ut1(category, timeout=300):
    return set(fetch_plain_list(UT1_BASE.format(category), timeout=timeout))


def fetch_adguard_phishing(timeout=120):
    r = requests.get(ADGUARD_PHISHING_URL, headers=HEADERS, timeout=timeout)
    r.raise_for_status()
    return {m.group(1) for m in
            (ADBLOCK_RULE.match(line.strip()) for line in r.text.splitlines()) if m}

In [ ]:
ut1_junk = {}
for _category in UT1_JUNK:
    ut1_junk[_category] = fetch_ut1(_category)
    print(f"ut1/{_category}: {len(ut1_junk[_category]):,}")

porn_domains = set()
for _url in PORN_URLS:
    porn_domains.update(fetch_hosts_file(_url))
print(f"porn: {len(porn_domains):,}")

adguard_phishing = fetch_adguard_phishing()
print(f"adguard/phishing: {len(adguard_phishing):,}")

ut1/gambling: 32,247
ut1/drogue: 603
ut1/cryptojacking: 11,491
ut1/ddos: 421
ut1/stalkerware: 525
ut1/malware: 252,120
porn: 74,472
adguard/phishing: 36,805


In [ ]:
class DomainMatcher:
    """Проверка вхождения домена в набор с учётом родительских доменов.

    Запись example.com в наборе считается покрывающей sub.example.com:
    поддомен фишингового хостинга остаётся фишинговым.
    """

    def __init__(self, domains):
        self.domains = set(domains)

    def match(self, domain):
        if domain in self.domains:
            return True
        labels = domain.split(".")
        return any(".".join(labels[i:]) in self.domains for i in range(1, len(labels)))

In [ ]:
feed_matcher = DomainMatcher(
    set().union(*ut1_junk.values()) | porn_domains | adguard_phishing)

by_feeds = {}
for _name in BRANCHES:
    _hits, _rest = [], []
    for d in branch[_name]:
        (_hits if feed_matcher.match(d) else _rest).append(d)
    by_feeds[_name] = _hits
    _share = len(_hits) / max(len(branch[_name]), 1)
    if _name == BLOCKED or FORBIDDEN_DROP:
        branch[_name] = _rest
        print(f"{_name}: отсеяно {len(_hits):,} ({_share:.2%}), "
              f"осталось {len(branch[_name]):,}")
    else:
        print(f"{_name}: найдено {len(_hits):,} ({_share:.2%}), "
              f"не удаляется (FORBIDDEN_DROP = False)")

# Находки в курируемой ветке печатаются целиком: их немного, и каждая —
# повод решить, ошибка это сборщика списка или ошибка фида.
print(f"\nнайдено фидами в {FORBIDDEN} ({len(by_feeds[FORBIDDEN]):,}):")
for _domain in by_feeds[FORBIDDEN]:
    print(f"  {_domain}")

ru-blocked: отсеяно 3,451 (15.38%), осталось 18,980
ru-forbidden: найдено 30 (1.83%), не удаляется (FORBIDDEN_DROP = False)

найдено фидами в ru-forbidden (30):
  1337x.to
  8chan.moe
  bitcoin.org
  chaturbate.com
  coomer.su
  e-hentai.org
  e621.net
  f95zone.to
  flexpool.io
  hashflare.io
  island-of-pleasure.site
  joyreactor.cc
  kemono.party
  kemono.su
  phncdn.com
  pornhub.com
  pornolab.net
  redtube.com
  rustorka.com
  rutor.info
  sex.com
  telegra.ph
  thepiratebay.org
  undress.cc
  xfantazy.com
  xhamster.com
  xhamsterlive.com
  xnxx.com
  xvideos.com
  static-ss.xvideos-cdn.com


## 7. Классификатор на символьных n-граммах — обе ветки

Второй слой отделения мусора от цензуры. Обучается на разметке UT1: положительный
класс — gambling, drogue, cryptojacking, ddos, stalkerware; отрицательный —
shopping, bank, press, jobsearch, games. Обе стороны из одного источника, поэтому
модель учится на различии категорий, а не на артефактах разных сборщиков.
Категория `games` в отрицательном классе обязательна: без неё модель принимает
игровые студии за гемблинг (`platinumgames.org` получал 0.998).

Выбор архитектуры. В задачах детекции вредоносных URL трансформеры дают лучший
результат — URLTran при доле ложных срабатываний 0.01% находит 86.8% вредоносных
адресов против 71.2% у свёрточной сети URLNet. Но там решается другая задача:
бинарная детекция свежего фишинга при экстремально низкой доле ложных
срабатываний, где выигрывает понимание контекста. Здесь разделяются тематические
категории, и сигнал буквально лексический — бренд-подстроки (`casino`, `vulkan`,
`1x`, `bet`) и их искажения, на которых n-граммы работают вровень. Модель
переобучается при каждом обновлении фидов, а раннеры GitHub Actions не имеют
видеокарт: десять секунд на процессоре против дообучения трансформера — решающий
аргумент при равном качестве.

**Порог.** Берётся из впадины плотности оценок: распределение двумодально —
легитимные жмутся к нулю, казино к единице, минимум плотности между модами и есть
точка разделения. Считается порог по выборке из ветки `ru-blocked` **до** фильтра
CrUX, а применяется к тому, что осталось **после** него и после разделов 4–6.
Причина в статистике: на широкой популяции реестра вторая мода населена и впадина
выражена, на нескольких тысячах преимущественно популярных доменов она
вырождается, и минимум плотности уезжает на край сетки. Считать плотность по всей
выгрузке незачем — оценка порога от этого не улучшится, поэтому берётся случайная
выборка фиксированного размера. Если проверка на выраженность впадины не проходит,
порог берётся с кривой точности-полноты по контрольной выборке UT1 — там, где
точность достигает 0.99.

**Защита.** Из правил защиты убрано «домен входит в топ-1M»: после фильтра CrUX
весь остаток `ru-blocked` популярен по построению, и это правило спасало ровно те
домены, ради которых слой существует (`1xbet55.com`, `azino777.com`). Остались
два правила: домен совпадает с публичным суффиксом и короткая метка второго
уровня — на таких n-граммам не за что зацепиться.

**Что остаётся ошибочным.** Болгарское издание `mediapool.bg` получает 0.999
из-за подстроки `pool`, характерной для майнинговых пулов. Модель работает с
именем домена и не может отличить такой случай без обращения к содержимому сайта.

На ветке `ru-forbidden` эта слабость видна отчётливо. Замер сквозного прогона:
выше порога оказывается около 6% ветки, и в верхушке списка — `4pda.to` (0.942),
`tiktokv.com` (0.937), `ytimg.com` (0.940), `adobe.io` (0.916),
`api.fitbit.com` (0.994). Модель обучена на именах казино и наркомагазинов и
реагирует на короткие метки с цифрами и на подстроки вроде `bi`, `to`, `fit`.
Поэтому по этой ветке печатается весь список превысивших порог, а удаление
управляется флагом `FORBIDDEN_DROP` из второй ячейки.

In [ ]:
ut1_legit = set()
for _category in UT1_LEGIT:
    ut1_legit |= fetch_ut1(_category)

# malware в обучение не идёт: там другая природа имён (взломанные легитимные
# сайты), и модель на них учится шуму. В слой фидов он при этом входит.
train_junk = sorted(
    set().union(*(ut1_junk[c] for c in
                  ["gambling", "drogue", "cryptojacking", "ddos", "stalkerware"]))
    - ut1_legit)
train_legit = sorted(ut1_legit)
print(f"обучающая выборка: мусор {len(train_junk):,}, легитимные {len(train_legit):,}")

обучающая выборка: мусор 45,034, легитимные 142,666


In [ ]:
X = train_junk + train_legit
y = np.array([1] * len(train_junk) + [0] * len(train_legit))
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

# char_wb ограничивает n-граммы границами меток домена: подстрока не
# «перетекает» через точку и остаётся признаком конкретной метки.
vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5),
                             min_df=3, sublinear_tf=True, max_features=300_000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

model = LogisticRegression(max_iter=3000, C=10, class_weight="balanced")
model.fit(X_train_vec, y_train)
print(f"признаков: {X_train_vec.shape[1]:,}")

test_scores = model.predict_proba(X_test_vec)[:, 1]
print(f"ROC-AUC: {roc_auc_score(y_test, test_scores):.4f}")
print(classification_report(y_test, test_scores > 0.5,
                            target_names=["легитимные", "мусор"], digits=3))

признаков: 206,516
ROC-AUC: 0.9799
              precision    recall  f1-score   support

  легитимные      0.973     0.963     0.968     35667
       мусор      0.886     0.916     0.901     11258

    accuracy                          0.952     46925
   macro avg      0.930     0.939     0.934     46925
weighted avg      0.952     0.952     0.952     46925



In [ ]:
MIN_PRECISION = 0.99
CALIBRATION_SAMPLE = 200_000


def threshold_by_precision(y_true, scores, min_precision=MIN_PRECISION):
    """Наименьший порог, при котором точность на контрольной выборке достигает цели."""
    precision, _, thresholds = precision_recall_curve(y_true, scores)
    ok = np.where(precision[:-1] >= min_precision)[0]
    return float(thresholds[ok[0]]) if len(ok) else 0.99


def threshold_by_valley(scores, lo=0.30, hi=0.95, depth=0.5):
    """Порог по минимуму плотности между модами. None, если впадины нет.

    Впадиной считается внутренний минимум, плотность в котором ниже
    указанной доли от меньшей из двух мод. Без этой проверки функция всегда
    что-нибудь возвращает — на одномодальном распределении просто край сетки.
    """
    if len(scores) < 500:
        return None
    kde = gaussian_kde(scores, bw_method=0.05)
    grid = np.linspace(lo, hi, 400)
    density = kde(grid)
    idx = int(np.argmin(density))
    if idx < 5 or idx > len(grid) - 6:
        return None
    mode_low = float(kde(np.linspace(0.0, lo, 100)).max())
    mode_high = float(kde(np.linspace(hi, 1.0, 100)).max())
    if density[idx] >= depth * min(mode_low, mode_high):
        return None
    return float(grid[idx])

In [ ]:
random.seed(42)
calibration_set = (random.sample(blocked_before_crux, CALIBRATION_SAMPLE)
                   if len(blocked_before_crux) > CALIBRATION_SAMPLE
                   else blocked_before_crux)
calibration_scores = model.predict_proba(vectorizer.transform(calibration_set))[:, 1]
print(f"калибровка по {len(calibration_set):,} доменам ветки {BLOCKED} до CrUX")

DROP_THRESHOLD = threshold_by_valley(calibration_scores)
if DROP_THRESHOLD is None:
    DROP_THRESHOLD = threshold_by_precision(y_test, test_scores)
    print(f"впадина не выражена, порог с кривой точности-полноты "
          f"(точность >= {MIN_PRECISION}): {DROP_THRESHOLD:.3f}")
else:
    print(f"порог по впадине плотности: {DROP_THRESHOLD:.3f}")

калибровка по 200,000 доменам ветки ru-blocked до CrUX
порог по впадине плотности: 0.701


In [ ]:
def protected(domain):
    """Причина, по которой домен не удаляется по решению модели."""
    if domain in PSL_RULES or domain in PSL_WILDCARDS:
        return "public-suffix"
    if len(registrable(domain).split(".")[0]) < 4:
        return "short-label"
    return None


def apply_model(domains):
    """Делит домены на оставленные, удалённые и защищённые."""
    if not domains:
        return [], [], []
    scores = model.predict_proba(vectorizer.transform(domains))[:, 1]
    kept, dropped, shielded = [], [], []
    for domain, score in zip(domains, scores):
        if score <= DROP_THRESHOLD:
            kept.append(domain)
        elif (reason := protected(domain)):
            shielded.append((domain, score, reason))
            kept.append(domain)
        else:
            dropped.append((domain, score))
    return kept, dropped, shielded

In [ ]:
by_model, shielded = {}, {}

for _name in BRANCHES:
    _kept, by_model[_name], shielded[_name] = apply_model(branch[_name])
    if _name == BLOCKED or FORBIDDEN_DROP:
        branch[_name] = _kept
        print(f"{_name}: удалено {len(by_model[_name]):,}, "
              f"защищено {len(shielded[_name]):,}, осталось {len(branch[_name]):,}")
    else:
        print(f"{_name}: набрали выше порога {len(by_model[_name]):,}, "
              f"не удаляются (FORBIDDEN_DROP = False)")

print(f"\nвыше порога в {BLOCKED}, топ-20 по баллу:")
for _domain, _score in sorted(by_model[BLOCKED], key=lambda t: -t[1])[:20]:
    print(f"  {_score:.3f}  {_domain}")

# По курируемой ветке — весь список: именно здесь ложное удаление дороже всего.
print(f"\nвыше порога в {FORBIDDEN} ({len(by_model[FORBIDDEN]):,}):")
for _domain, _score in sorted(by_model[FORBIDDEN], key=lambda t: -t[1]):
    print(f"  {_score:.3f}  {_domain}")

ru-blocked: удалено 2,754, защищено 184, осталось 16,226
ru-forbidden: набрали выше порога 120, не удаляются (FORBIDDEN_DROP = False)

выше порога в ru-blocked, топ-20 по баллу:
  1.000  vulkan-casino-slotss.com
  1.000  vulkancasinoss.net.ru
  1.000  vulkan777casino-online.org
  1.000  1xcasino-st.com
  1.000  vulkan777casino-online.net
  1.000  pokerdom-casino-n5r.top
  1.000  vulkan777-casinosite.club
  1.000  0normcasino-ff.win
  1.000  1normcasino-xa.win
  1.000  1xcasino.com
  1.000  vulkandeluxe-casino.club
  1.000  gambling-casino2.lol
  1.000  vulkan24casino-site.biz
  1.000  5normcasino-of.win
  1.000  0normcasino-xe.win
  1.000  9normcasino-of.win
  1.000  1normcasino-xx.win
  1.000  vulkanvegas.com
  1.000  2normcasino-oe.win
  1.000  vulkan24-casinoonline.net

выше порога в ru-forbidden (120):
  0.994  api.fitbit.com
  0.994  flexpool.io
  0.993  together.ai
  0.992  bitcoin.org
  0.987  geospy.ai
  0.983  xnxx.com
  0.977  xn--mts47c3w9b1qr.cn
  0.972  hashflare.io
  0.95

## 8. Google Safe Browsing — обе ветки

Последний слой отсева мусора: база Google обновляется чаще любого из ручных фидов.
Шаг выполняется, только если задан ключ API. Раньше он был непозволительно дорог —
проверять приходилось сотни тысяч доменов, и результат приходилось фиксировать
вручную; после разделов 3–7 проверяются тысячи, то есть единицы запросов по 500
адресов, и слой работает в каждой сборке.

In [ ]:
SAFE_BROWSING_URL = "https://safebrowsing.googleapis.com/v4/threatMatches:find"

try:
    from google.colab import userdata
    SAFE_BROWSING_KEY = userdata.get("GOOGLE_SAFE_BROWSING_KEY")
except Exception:
    SAFE_BROWSING_KEY = os.environ.get("GOOGLE_SAFE_BROWSING_KEY")


def safe_browsing_flags(domains, api_key, batch_size=500, desc="Safe Browsing"):
    flagged = set()
    for start in tqdm(range(0, len(domains), batch_size), desc=desc):
        batch = domains[start:start + batch_size]
        body = {
            "client": {"clientId": "geosite-ru-builder", "clientVersion": "1.0"},
            "threatInfo": {
                "threatTypes": ["SOCIAL_ENGINEERING", "MALWARE", "UNWANTED_SOFTWARE"],
                "platformTypes": ["ANY_PLATFORM"],
                "threatEntryTypes": ["URL"],
                "threatEntries": [{"url": f"http://{d}"} for d in batch],
            },
        }
        r = requests.post(SAFE_BROWSING_URL, params={"key": api_key},
                          json=body, timeout=60)
        r.raise_for_status()
        for match in r.json().get("matches", []):
            host = urlsplit(match["threat"]["url"]).hostname or ""
            flagged.add(host.lower().strip("."))
        time.sleep(1)
    return flagged

In [ ]:
if SAFE_BROWSING_KEY:
    for _name in BRANCHES:
        _flagged = safe_browsing_flags(branch[_name], SAFE_BROWSING_KEY,
                                       desc=f"Safe Browsing, {_name}")
        if _name == BLOCKED or FORBIDDEN_DROP:
            _before = len(branch[_name])
            branch[_name] = [d for d in branch[_name] if d not in _flagged]
            print(f"{_name}: помечено {len(_flagged):,}, "
                  f"удалено {_before - len(branch[_name]):,}, "
                  f"осталось {len(branch[_name]):,}")
        else:
            print(f"{_name}: помечено {len(_flagged):,}, "
                  f"не удаляется (FORBIDDEN_DROP = False)")
            for _domain in sorted(_flagged):
                print(f"  {_domain}")
else:
    print("ключ GOOGLE_SAFE_BROWSING_KEY не задан, слой пропущен")

Safe Browsing, ru-blocked: 100%|██████████| 33/33 [00:33<00:00,  1.03s/it]


ru-blocked: помечено 0, удалено 0, осталось 16,226


Safe Browsing, ru-forbidden: 100%|██████████| 4/4 [00:04<00:00,  1.02s/it]

ru-forbidden: помечено 0, не удаляется (FORBIDDEN_DROP = False)


## 9. Сборка geosite.dat

Файл пишется напрямую в protobuf. Официальный компилятор `domain-list-community`
требует Go >= 1.25, которого нет в пакетах Ubuntu — установка тулчейна в CI ради
сериализации трёх вложенных сообщений не оправдана.

Схема из `app/router/config.proto` v2ray-core:

```
message Domain {
  enum Type { Plain = 0; Regex = 1; Domain = 2; Full = 3; }
  Type   type  = 1;
  string value = 2;
}
message GeoSite     { string country_code = 1; repeated Domain domain = 2; }
message GeoSiteList { repeated GeoSite entry = 1; }
```

Списки пишутся раздельно, но пересечение между ними снимается: домен, попавший в
обе ветки, остаётся только в `ru-forbidden`. Маршрут для обеих один, поэтому
вторая запись ничего не добавляет, а память клиента расходует. Приоритет отдан
`ru-forbidden` потому, что эта ветка курируемая и её содержимое проверено людьми.

Запись проверяется обратным разбором: кодировщик самодельный, и молчаливая
ошибка в нём проявилась бы уже на телефоне.

In [22]:
TYPE_DOMAIN = 2  # правило с покрытием поддоменов


def _varint(value):
    out = bytearray()
    while True:
        byte = value & 0x7F
        value >>= 7
        out.append(byte | 0x80 if value else byte)
        if not value:
            return bytes(out)


def _tag(field, wire):
    return _varint((field << 3) | wire)


def _len_delim(field, payload):
    return _tag(field, 2) + _varint(len(payload)) + payload


def encode_geosite(lists):
    """{имя списка: [домен, ...]} → содержимое geosite.dat.

    Имена приводятся к верхнему регистру: так они хранятся во всех
    публикуемых .dat, обращение geosite:ru-blocked регистронезависимо.
    """
    out = bytearray()
    for name, domains in lists.items():
        entry = _len_delim(1, name.upper().encode())
        for domain in domains:
            rule = _tag(1, 0) + _varint(TYPE_DOMAIN) + _len_delim(2, domain.encode())
            entry += _len_delim(2, rule)
        out += _len_delim(1, entry)
    return bytes(out)

In [13]:
def _read_varint(buf, pos):
    result = shift = 0
    while True:
        byte = buf[pos]
        pos += 1
        result |= (byte & 0x7F) << shift
        if not byte & 0x80:
            return result, pos
        shift += 7


def _iter_fields(buf):
    pos = 0
    while pos < len(buf):
        key, pos = _read_varint(buf, pos)
        field, wire = key >> 3, key & 7
        if wire == 0:
            value, pos = _read_varint(buf, pos)
            yield field, value
        elif wire == 2:
            length, pos = _read_varint(buf, pos)
            yield field, buf[pos:pos + length]
            pos += length
        else:
            raise ValueError(f"неподдерживаемый wire type {wire}")


def decode_geosite(raw_bytes):
    """Обратный разбор — используется для проверки записанного файла."""
    result = {}
    for field, payload in _iter_fields(raw_bytes):
        if field != 1:
            continue
        name, domains = None, []
        for sub_field, sub_payload in _iter_fields(payload):
            if sub_field == 1:
                name = sub_payload.decode()
            elif sub_field == 2:
                value = None
                for dom_field, dom_payload in _iter_fields(sub_payload):
                    if dom_field == 2:
                        value = dom_payload.decode()
                if value:
                    domains.append(value)
        if name:
            result[name] = domains
    return result

In [ ]:
# Повторная дедупликация внутри веток: предыдущие слои могли удалить
# родительский домен, оставив его поддомены непокрытыми.
ru_forbidden = dedup_by_coverage(branch[FORBIDDEN])

_forbidden_matcher = DomainMatcher(ru_forbidden)
ru_blocked = [d for d in dedup_by_coverage(branch[BLOCKED])
              if not _forbidden_matcher.match(d)]

_total = len(ru_blocked) + len(ru_forbidden)
print(f"{BLOCKED}:   {len(ru_blocked):,}")
print(f"{FORBIDDEN}: {len(ru_forbidden):,}")
print(f"итого:        {_total:,} / {MAX_DOMAINS:,}")

if _total > MAX_DOMAINS:
    print(f"\nПРЕВЫШЕН БЮДЖЕТ на {_total - MAX_DOMAINS:,}")
    print("пороги CRUX_RANK_MAX и суммарный размер, который каждый оставляет:")
    for _bucket in CRUX_BUCKETS:
        _count = sum(1 for d in ru_blocked
                     if CRUX_RANK.get(registrable(d), CRUX_RANK_MAX) <= _bucket)
        _sum = _count + len(ru_forbidden)
        _mark = "  <- укладывается" if _sum <= MAX_DOMAINS else ""
        print(f"  rank <= {_bucket:>9,}: {_count:>7,} + {len(ru_forbidden):,} "
              f"= {_sum:>7,}{_mark}")
    print("списки записываются целиком; чтобы сократить — уменьшить "
          "CRUX_RANK_MAX и перезапустить с раздела 3")

ru-blocked:   16,047
ru-forbidden: 1,639
итого:        17,686 / 10,000

ПРЕВЫШЕН БЮДЖЕТ на 7,686
пороги CRUX_RANK_MAX и суммарный размер, который каждый оставляет:
  rank <=     1,000:     289 + 1,639 =   1,928  <- укладывается
  rank <=     5,000:     602 + 1,639 =   2,241  <- укладывается
  rank <=    10,000:     874 + 1,639 =   2,513  <- укладывается
  rank <=    50,000:   2,974 + 1,639 =   4,613  <- укладывается
  rank <=   100,000:   5,989 + 1,639 =   7,628  <- укладывается
  rank <=   500,000:  12,128 + 1,639 =  13,767
  rank <= 1,000,000:  16,047 + 1,639 =  17,686
списки записываются целиком; чтобы сократить — уменьшить CRUX_RANK_MAX и перезапустить с раздела 3


In [ ]:
os.makedirs("publish", exist_ok=True)

with open("publish/geosite.dat", "wb") as f:
    f.write(encode_geosite({BLOCKED: ru_blocked, FORBIDDEN: ru_forbidden}))

for _name, _domains in ((BLOCKED, ru_blocked), (FORBIDDEN, ru_forbidden)):
    with open(f"publish/{_name}.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(_domains) + "\n")

print(sorted(os.listdir("publish")))

['geosite.dat', 'ru-blocked.txt', 'ru-forbidden.txt']


In [ ]:
raw_dat = open("publish/geosite.dat", "rb").read()
parsed = decode_geosite(raw_dat)

assert set(parsed) == {BLOCKED.upper(), FORBIDDEN.upper()}
assert parsed[BLOCKED.upper()] == ru_blocked
assert parsed[FORBIDDEN.upper()] == ru_forbidden

_size_mb = len(raw_dat) / 1024 / 1024
_total = len(ru_blocked) + len(ru_forbidden)
print("проверка разбором пройдена:", {k: f"{len(v):,}" for k, v in parsed.items()})
print(f"размер geosite.dat: {_size_mb:.2f} МБ")
print(f"бюджет: {_total:,} / {MAX_DOMAINS:,} "
      f"({'в пределах' if _total <= MAX_DOMAINS else 'ПРЕВЫШЕН'})")

проверка разбором пройдена: {'RU-BLOCKED': '16,047', 'RU-FORBIDDEN': '1,639'}
размер geosite.dat: 0.43 МБ
бюджет: 17,686 / 10,000 (ПРЕВЫШЕН)


## 10. geoip.dat

Доменное правило работает, когда в соединение попало имя: в TLS его достаёт
сниффер из поля SNI, в HTTP — из заголовка. Когда приложение подключается сразу
по адресу, имени нет нигде, и `geosite` сопоставлять не с чем. Так работает
MTProto у Telegram: это не TLS, поля с именем в протоколе нет вовсе, а адреса
дата-центров зашиты в клиент. Ни одна запись в `geosite.dat` такое соединение не
перехватит — нужно правило по адресам.

Собирать диапазоны разрешением доменов в A-записи нельзя: замер на выборке из
1200 доменов ветки `ru-blocked` дал 944 уникальных адреса, из которых 54.9%
принадлежат Cloudflare, а 40 доменов разрешились в `127.0.0.1`. Правило на
anycast-адрес сети доставки — это правило на всех её арендаторов.

Работающий способ — взять готовые категории из чужого `geoip.dat`, где диапазоны
собраны по автономным системам самих сервисов. Совпадение ищется по бренду:
метка перед публичным суффиксом (`netflix.com` → `netflix`) сверяется с именем
категории. Точное совпадение, без таблицы синонимов: `nflxvideo.net` и
`youtube.com` не совпадут, и это осознанный размен на отсутствие ручного
сопровождения.

Источник — `Loyalsoldier/v2ray-rules-dat`. Релизы самого `v2fly/geoip` не
подходят: там 253 категории, из них 251 код страны плюс `PRIVATE` и `TEST`,
сервисных категорий нет.

Домены, вызвавшие совпадение, из `geosite.dat` **не** удаляются. Доменное правило
работает там, где имя есть, адресное — там, где его нет; убрать первое означает
сломать маршрутизацию в момент, когда список адресов устареет.

In [26]:
# Сборка с сервисными категориями. Релизы v2fly/geoip не годятся: 253 категории,
# из них 251 код страны плюс PRIVATE и TEST.
FOREIGN_GEOIP_URL = "https://github.com/Loyalsoldier/v2ray-rules-dat/releases/latest/download/geoip.dat"

# Общие сети доставки: их диапазоны делят миллионы посторонних сайтов.
# Отбрасываются независимо от совпадения.
GEOIP_DENY = {"GOOGLE", "CLOUDFLARE", "CLOUDFRONT", "FASTLY", "AKAMAI"}

# Сопоставлять ли бренды с двухбуквенными кодами стран. Замер на релизе
# 202608241843: включение даёт 116 166 префиксов из случайных совпадений —
# fr.de тянет всю Францию (40 248), se.com (Schneider Electric) — Швецию
# (13 793), ni.com (National Instruments) — Никарагуа, а 11ebalka.ru.actor —
# всю Россию (25 138), то есть весь российский трафик уходит в прокси.
# На не страновых категориях ложных срабатываний нет.
GEOIP_MATCH_COUNTRIES = False

# Добавляется всегда: без него локальные адреса и петлевой интерфейс
# могут уйти в прокси.
GEOIP_ALWAYS = {"PRIVATE"}


def to_cidr(value):
    """Строка вида 1.2.3.0/24 → (сырые байты адреса, длина префикса)."""
    net = ipaddress.ip_network(value, strict=False)
    return net.network_address.packed, net.prefixlen


def cidr_str(packed, prefix):
    return f"{ipaddress.ip_address(packed)}/{prefix}"


def encode_geoip(lists):
    """{имя категории: [(сырые байты, длина префикса), ...]} → содержимое geoip.dat.

    Схема из common/geodata/geodat.proto: CIDR{bytes ip = 1; uint32 prefix = 2},
    GeoIP{string code = 1; repeated CIDR cidr = 2}, GeoIPList{repeated GeoIP = 1}.
    Адрес хранится сырыми байтами — четыре для IPv4, шестнадцать для IPv6, —
    а не строкой, поэтому файл выходит примерно вдвое компактнее geosite
    в пересчёте на запись.
    """
    out = bytearray()
    for code, networks in lists.items():
        entry = _len_delim(1, code.upper().encode())
        for packed, prefix in networks:
            cidr = _len_delim(1, packed) + _tag(2, 0) + _varint(prefix)
            entry += _len_delim(2, cidr)
        out += _len_delim(1, entry)
    return bytes(out)


def decode_geoip(raw_bytes):
    """Обратный разбор; значения остаются парами (байты, длина префикса)."""
    result = {}
    for field, payload in _iter_fields(raw_bytes):
        if field != 1:
            continue
        code, networks = None, []
        for sub_field, sub_payload in _iter_fields(payload):
            if sub_field == 1:
                code = sub_payload.decode()
            elif sub_field == 2:
                packed = prefix = None
                for c_field, c_payload in _iter_fields(sub_payload):
                    if c_field == 1:
                        packed = c_payload
                    elif c_field == 2:
                        prefix = c_payload
                if packed is not None and prefix is not None:
                    networks.append((packed, prefix))
        if code:
            result[code] = networks
    return result

In [27]:
_resp = requests.get(FOREIGN_GEOIP_URL, headers=HEADERS, timeout=300)
_resp.raise_for_status()
foreign_geoip = decode_geoip(_resp.content)

# Двухбуквенный код — страна; всё остальное собрано по автономным системам
# сервисов. Деление по длине имени, потому что иного признака в формате нет.
service_cats = {c: n for c, n in foreign_geoip.items() if len(c) > 2}

print(f"скачано {len(_resp.content) / 1024 / 1024:.1f} МБ, "
      f"категорий {len(foreign_geoip)}, из них сервисных {len(service_cats)}")
for _code, _nets in sorted(service_cats.items(), key=lambda t: -len(t[1])):
    _mark = "  (в запретном списке)" if _code in GEOIP_DENY else ""
    print(f"  {_code:12s} {len(_nets):>7,}{_mark}")

скачано 16.4 МБ, категорий 260, из них сервисных 10
  GOOGLE         7,947  (в запретном списке)
  TOR              958
  CLOUDFLARE       672  (в запретном списке)
  CLOUDFRONT       211  (в запретном списке)
  FACEBOOK         122
  NETFLIX          114
  FASTLY            93  (в запретном списке)
  TWITTER           20
  PRIVATE           18
  TELEGRAM          12


In [28]:
def brand(domain):
    """Метка перед публичным суффиксом: sub.netflix.co.uk → netflix."""
    return registrable(domain).split(".")[0]


_pool = dict(foreign_geoip) if GEOIP_MATCH_COUNTRIES else dict(service_cats)
_by_lower = {c.lower(): c for c in _pool}

matched = {}
for _name in BRANCHES:
    for _domain in (ru_blocked if _name == BLOCKED else ru_forbidden):
        _code = _by_lower.get(brand(_domain))
        if _code and _code not in GEOIP_DENY:
            matched.setdefault(_code, []).append(_domain)

print(f"совпало категорий: {len(matched)}")
for _code, _domains in sorted(matched.items(), key=lambda t: -len(_pool[t[0]])):
    print(f"  {_code:12s} префиксов {len(_pool[_code]):>7,}  "
          f"вызвано: {', '.join(sorted(_domains)[:5])}"
          f"{' …' if len(_domains) > 5 else ''}")

_denied = {brand(d) for lst in (ru_blocked, ru_forbidden) for d in lst
           if _by_lower.get(brand(d)) in GEOIP_DENY}
if _denied:
    print(f"\nотброшено по запретному списку: {', '.join(sorted(_denied))}")

совпало категорий: 4
  FACEBOOK     префиксов     122  вызвано: facebook.com
  NETFLIX      префиксов     114  вызвано: netflix.ca, netflix.com, netflix.com.au, netflix.net
  TWITTER      префиксов      20  вызвано: twitter.com
  TELEGRAM     префиксов      12  вызвано: telegram.app, telegram.com, telegram.dev, telegram.dog, telegram.me …

отброшено по запретному списку: cloudflare, google


In [30]:
geoip_lists = {code: _pool[code] for code in sorted(matched)}
for _code in GEOIP_ALWAYS:
    if _code in foreign_geoip:
        geoip_lists[_code] = foreign_geoip[_code]
    else:
        print(f"ВНИМАНИЕ: категории {_code} нет в источнике")

_blob = encode_geoip(geoip_lists)
with open("publish/geoip.dat", "wb") as f:
    f.write(_blob)

# Проверка обратным разбором: кодировщик самодельный, и молчаливая ошибка
# в нём проявилась бы уже на телефоне.
_parsed = decode_geoip(open("publish/geoip.dat", "rb").read())
assert _parsed == geoip_lists

_total = sum(len(v) for v in geoip_lists.values())
print(f"geoip.dat: {len(_blob) / 1024:.1f} КБ, категорий {len(geoip_lists)}, "
      f"префиксов {_total:,}")
for _code, _nets in sorted(geoip_lists.items(), key=lambda t: -len(t[1])):
    print(f"  {_code:12s} {len(_nets):>7,}  "
          f"{cidr_str(*_nets[0])} … {cidr_str(*_nets[-1])}")

geoip.dat: 4.4 КБ, категорий 5, префиксов 286
  FACEBOOK         122  31.13.24.0/21 … 2c0f:ef78:10::/47
  NETFLIX          114  23.246.0.0/18 … 2a03:5640:f800::/37
  TWITTER           20  8.25.194.0/23 … 2a04:9d40:f000::/36
  PRIVATE           18  0.0.0.0/8 … ff00::/8
  TELEGRAM          12  91.105.192.0/23 … 2a0a:f280::/32
